# MCU-RAG on Google Colab / Jupyter

**Stop scrolling datasheets. Start shipping firmware.**

Run the fully-local pixel-RAG pipeline on a Colab GPU. This notebook installs Ollama, pulls the vision model (`qwen3-vl:8b`) and the text embedder (`nomic-embed-text`), then lets you ingest any datasheet PDF and chat with it.

> **Tip:** Runtime → Change runtime type → **GPU** (a T4 has 16 GB and fits the model fully, so it is much faster than an 8 GB laptop).

## 1. Install Ollama and start the server

In [ ]:
import subprocess, time, requests

# Install Ollama (Linux one-liner)
subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=True)

# Start the Ollama server in the background
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Wait until it answers
for _ in range(60):
    try:
        if requests.get('http://localhost:11434/api/version', timeout=2).ok:
            print('Ollama up:', requests.get('http://localhost:11434/api/version').json())
            break
    except Exception:
        time.sleep(1)
else:
    print('Ollama did not start - check the runtime')

## 2. Pull the models (one-time, ~6 GB)

In [ ]:
!ollama pull qwen3-vl:8b
!ollama pull nomic-embed-text

## 3. Get the code and install Python deps

Edit `REPO_URL` to point at your fork/repo.

In [ ]:
REPO_URL = 'https://github.com/s1d-r/MCU-RAG.git'  # <-- EDIT ME

import os
if not os.path.isdir('mcu-rag'):
    !git clone $REPO_URL mcu-rag
%cd mcu-rag
!pip -q install -r requirements.txt

## 4. Tune for the Colab GPU

A 16 GB GPU fits the model fully, so we can use bigger images and more pages than on an 8 GB laptop. (Set these **before** importing the modules.)

In [ ]:
import os
os.environ['PIXELRAG_TOP_K'] = '3'
os.environ['PIXELRAG_MAX_IMAGE_PX'] = '1600'
os.environ['PIXELRAG_NUM_CTX'] = '8192'

## 5. Upload a datasheet PDF and build the index

In [ ]:
from google.colab import files
uploaded = files.upload()           # pick a datasheet PDF
pdf_path = list(uploaded.keys())[0]

import ingest
index_dir = ingest.build_index(pdf_path, out_dir='colab_index')
print('Index built at', index_dir)

## 6. Ask questions (the reader looks at the page images)

In [ ]:
import ask as ask_mod

question = 'Which pin is the user LED connected to?'  # <-- change me
res = ask_mod.ask(question, 'colab_index')
print(res['answer'])
print('\nretrieved pages:', res['pages_used'])

## 7. (Optional) Launch the full web UI via a free Cloudflare tunnel

This starts the FastAPI app and prints a public `https://*.trycloudflare.com` URL. Open it, drop a PDF, and chat with the fancy UI — all running on the Colab GPU.

In [ ]:
import subprocess, time

# Start the server
subprocess.Popen(['python', 'server.py'])
time.sleep(4)

# Download cloudflared and open a quick tunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
proc = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='')
    if 'trycloudflare.com' in line:
        print('\n>>> Open the trycloudflare.com URL above in your browser <<<')
        break